<a href="https://colab.research.google.com/github/Aggarwalmansi/GENAI/blob/main/AgenticAIClass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.2 MB/s eta 0:00:00


In [2]:
import os #handles files
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from google.colab import userdata #bring the api key from secrets

In [4]:
my_api_key = userdata.get('groq_api')

In [5]:
llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = my_api_key,
    temperature= 0
)

In [14]:
num1 = 952124
num2 = 123991
# query = f'What is {num1} * {num2}? Only give me numeric answer?'
query = 'what is capital of india'
response = llm.invoke(query)
print(response)

content='The capital of India is **New Delhi**.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 40, 'total_tokens': 51, 'completion_time': 0.020942142, 'completion_tokens_details': None, 'prompt_time': 0.004946796, 'prompt_tokens_details': None, 'queue_time': 0.077778217, 'total_time': 0.025888938}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_e65acd3773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d3ce6-38a9-7e20-a1c4-38a75a7f8dc8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 11, 'total_tokens': 51}


In [15]:
def multiply(a:int,b:int) -> int:
  """
    Multiplies two intergers together.
    Use this tool whenever you need to perform multiplication.
  """
  print("Multiplication tool called")
  return a*b

In [49]:
def divide(a:int , b:int) -> int:
  """
  Divides two integers together.
  Use this tool whenever you need to perform division.
  """
  print("Division tool called")
  return a/b

In [76]:
def addition(a:int,b:int) -> int :
  """
  Add two integers together.
  Use this tool whenever you need to perform addition.
  """
  print("Addition tool called")
  return a+b

In [80]:
llm_with_tools = llm.bind_tools([multiply,divide,addition])

In [81]:
response = llm_with_tools.invoke(query)
print(response.content)

The capital of India is New Delhi.


In [82]:
print(response.tool_calls)

[]


In [83]:
!pip install langgraph

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [84]:
class State(TypedDict):
  messages: Annotated[list, add_messages]

In [96]:
tools = [multiply,divide,addition]

In [86]:
# Input: The current State
# Output: A dictionary with the NEW message to add to State
def assistant_node(state: State):
    # Get the conversation history
    current_messages = state["messages"]

    # Ask the LLM what to do
    response = llm_with_tools.invoke(current_messages)
    # Return the response so it gets added to the 'messages' list
    return {"messages": [response]}

In [87]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)
# Because currently we have one tool, what we have 100 tools?
# You don't need to create 100 nodes, this code can handle all tools

In [88]:
from typing import Literal
from langgraph.graph import END

def should_continue(state: State) -> Literal['tools', END]:
    messages = state['messages']
    last_message = messages[-1]

    if last_message.tool_calls:
      return 'tools'
    else:
      return END

In [89]:
from langgraph.graph import StateGraph, START

# 1. Create a blank graph
builder = StateGraph(State)

In [90]:
builder.add_node('assistant', assistant_node)
builder.add_node('tools', tool_node)

In [91]:
builder.add_edge(START, 'assistant')

In [92]:
builder.add_conditional_edges(
    'assistant',
    should_continue,

)

In [93]:
builder.add_edge('tools', 'assistant')

In [94]:
graph = builder.compile()

In [95]:
# 1. Define the user's complex request
user_query = "Calculate 952124 divide by 123991. Tell me the answer"

# 2. Run the Graph
# We stream the steps so we can see the "thinking" process live
initial_state = {"messages": [("user", user_query)]}

print(f"User: {user_query}\n")
print("--- STARTING AGENT LOOP ---")

for step in graph.stream(initial_state):
    # Print the name of the node that just finished running
    node_name = list(step.keys())[0]
    print(f"📍 Node '{node_name}' finished.")

    # Optional: Print the actual output to see what happened inside
    # print(step)

print("--- END ---")

# 3. Print the final result
final_response = graph.invoke(initial_state)
print("\n🤖 Final Answer:")
print(final_response["messages"][-1].content)

User: Calculate 952124 divide by 123991. Tell me the answer

--- STARTING AGENT LOOP ---
📍 Node 'assistant' finished.
Division tool called
📍 Node 'tools' finished.
📍 Node 'assistant' finished.
--- END ---
Division tool called

🤖 Final Answer:
The answer is 7.678976699921768.
